<a href="https://colab.research.google.com/github/Diego-galsan/IA_competitions/blob/kaggle_llm_finetunning/Kaggle_competition_deberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import pandas as pd

## Using a BERT (DeBERTa) Model for Classification

The following cells demonstrate how to use a BERT-based model (specifically DeBERTa, which is often superior for NLU tasks) using the Hugging Face `transformers` library. We will load the competition data, preprocess it, and run a forward pass with a pre-trained model.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

# Check if MPS (Metal Performance Shaders) is available for Mac, otherwise use CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [4]:
# Load the training data
train_path = "train.csv"
df = pd.read_csv(train_path)# Display the first few rows
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [5]:
# Load Tokenizer and Model
# We use DeBERTa-v3-small as it is efficient and powerful.
# For better performance in competition, consider 'microsoft/deberta-v3-base' or 'large'.
model_name = "microsoft/deberta-v3-small"

# Use DebertaV2Tokenizer directly to avoid AutoTokenizer issues with fast/slow conversion
from transformers import DebertaV2Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3) # 3 labels: A wins, B wins, Tie

model.to(device)
print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully


In [6]:
# Example Inference
row = df.iloc[0]

# The columns 'prompt', 'response_a', 'response_b' might be JSON strings of lists (conversation turns)
# We'll just join them for this example
def process_text(text):
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return "\n".join(data)
        return text
    except:
        return text

prompt = process_text(row['prompt'])
response_a = process_text(row['response_a'])
response_b = process_text(row['response_b'])

# Construct input for the model: [CLS] prompt [SEP] response_a [SEP] response_b [SEP]
# DeBERTa tokenizer handles the special tokens automatically when passing multiple sequences,
# but since we have 3 parts, we can concatenate them manually or use the tokenizer's text_pair functionality.
# A common approach for this task is: Prompt + Response A + Response B
input_text = f"Prompt: {prompt}\n\nResponse A: {response_a}\n\nResponse B: {response_b}"

inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)

print(f"Logits: {logits}")
print(f"Probabilities (A wins, B wins, Tie): {probabilities}")

Logits: tensor([[-0.2406,  0.2606, -0.0366]])
Probabilities (A wins, B wins, Tie): tensor([[0.2579, 0.4258, 0.3163]])


## Fine-Tuning the Model

The model loaded above is a "base" model. It understands English but doesn't know how to classify "winner_model_a", "winner_model_b", or "tie" yet. We need to fine-tune it on our dataset.

We will:
1. Create a Hugging Face `Dataset`.
2. Tokenize the entire dataset.
3. Set up a `Trainer` to train the model.

In [7]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import numpy as np

# 1. Prepare the Data
# We need to map the labels to integers: A wins -> 0, B wins -> 1, Tie -> 2
def get_label(row):
    if row['winner_model_a'] == 1:
        return 0
    elif row['winner_model_b'] == 1:
        return 1
    else:
        return 2


df['label'] = df.apply(get_label, axis=1)

# Create a Hugging Face Dataset
dataset = Dataset.from_pandas(df[['prompt', 'response_a', 'response_b', 'label']])

# 2. Tokenization Function
def tokenize_function(examples):
    # We need to process the text lists just like before
    prompts = [process_text(t) for t in examples['prompt']]
    responses_a = [process_text(t) for t in examples['response_a']]
    responses_b = [process_text(t) for t in examples['response_b']]

    # Create the input text: Prompt + Response A + Response B
    inputs = [f"Prompt: {p}\n\nResponse A: {ra}\n\nResponse B: {rb}" for p, ra, rb in zip(prompts, responses_a, responses_b)]

    return tokenizer(inputs, padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Split into train and validation
# Using a small subset for demonstration speed. Use larger split for real training.
train_test_split = tokenized_datasets.train_test_split(test_size=0.1)
#train_dataset = train_test_split['train'].select(range(1000))
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print("Data prepared for training")

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

Data prepared for training


In [ ]:
# 3. Training Setup
# Full training setup with Trainer API
# Define metrics to evaluate performance
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",     # Save model at the end of each epoch
    learning_rate=2e-5,        # Standard learning rate for fine-tuning
    per_device_train_batch_size=4, # Adjust based on your VRAM (4 or 8 is usually safe for small models)
    per_device_eval_batch_size=4,
    num_train_epochs=1,        # Start with 1 epoch to test
    weight_decay=0.01,
    logging_steps=10,
    use_mps_device=True if device.type == 'mps' else False # Use MPS for Mac acceleration if available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# Start Training
print("Starting training...")
trainer.train()

/Users/diegogalvan/Documents/ia-laboratory/ai-tests/lib/python3.14/site-packages/transformers/training_args.py:2301: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(
/Users/diegogalvan/Documents/ia-laboratory/ai-tests/lib/python3.14/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.094200,1.096805,0.346903


TrainOutput(global_step=250, training_loss=1.1030707550048828, metrics={'train_runtime': 718.7929, 'train_samples_per_second': 1.391, 'train_steps_per_second': 0.348, 'total_flos': 132474479616000.0, 'train_loss': 1.1030707550048828, 'epoch': 1.0})

# LoRA

In [8]:
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj", "key_proj"],
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.to(device)
print("LoRA model loaded successfully")

trainable params: 444,675 || all params: 142,341,894 || trainable%: 0.3124
LoRA model loaded successfully


In [9]:
# 3. Training Setup with LoRA (with compatibility fix)

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Remove the problematic argument
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": (predictions == labels).mean()}

training_args = TrainingArguments(
    output_dir="./results_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    learning_rate=1e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,  # Important for PEFT
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("Starting LoRA training...")
trainer.train()

Starting LoRA training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Accuracy
1,1.067800,1.075179,0.399965
2,1.029700,1.074788,0.404315
3,1.060000,1.071144,0.413361


TrainOutput(global_step=19401, training_loss=1.078338717020736, metrics={'train_runtime': 4698.8651, 'train_samples_per_second': 33.026, 'train_steps_per_second': 4.129, 'total_flos': 2.077030896594739e+16, 'train_loss': 1.078338717020736, 'epoch': 3.0})

# Predictions

In [10]:
# Load test data
test_path = "test.csv"
test_df = pd.read_csv(test_path)

print(f"Test samples: {len(test_df)}")
test_df.head()

Test samples: 3


,id,prompt,response_a,response_b
0,136060,"[""I have three oranges today, I ate an orange ...","[""You have two oranges today.""]","[""You still have three oranges. Eating an oran..."
1,211333,"[""You are a mediator in a heated political deb...","[""Thank you for sharing the details of the sit...","[""Mr Reddy and Ms Blue both have valid points ..."
2,1233961,"[""How to initialize the classification head wh...","[""When you want to initialize the classificati...","[""To initialize the classification head when p..."


In [17]:
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm

def predict_probabilities(model, tokenizer, df, batch_size=16):
    # 1. Auto-detect device
    device = next(model.parameters()).device
    model.eval()

    all_probs = []

    # 2. Pre-format text
    print("Formatting text data...")
    formatted_texts = []
    for _, row in df.iterrows():
        # Ensure your text columns match your dataframe
        prompt = row['prompt']
        res_a = row['response_a']
        res_b = row['response_b']
        formatted_texts.append(
            f"Prompt: {prompt}\n\nResponse A: {res_a}\n\nResponse B: {res_b}"
        )

    # 3. Inference Loop
    print(f"Starting inference with batch_size={batch_size}...")
    with torch.no_grad():
        for i in tqdm(range(0, len(formatted_texts), batch_size)):
            batch_texts = formatted_texts[i : i + batch_size]

            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512
            )

            # Move inputs to GPU
            inputs = {key: val.to(device) for key, val in inputs.items()}

            outputs = model(**inputs)
            logits = outputs.logits

            # --- CHANGE IS HERE ---
            # Instead of argmax, we use Softmax to get probabilities (0.0 to 1.0)
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.extend(probs)

    return all_probs

# 1. Run inference
model_probs = predict_probabilities(model, tokenizer, test_df, batch_size=16)

# 2. Create the submission DataFrame
# Assuming the model outputs are ordered: [Index 0=Model A, Index 1=Model B, Index 2=Tie]
submission = pd.DataFrame(model_probs, columns=['winner_model_a', 'winner_model_b', 'winner_tie'])

# 3. Add the ID column from the original test_df
submission.insert(0, 'id', test_df['id'])

# 4. Check the format
print(submission.head())

# 5. Save to CSV
submission.to_csv('submission.csv', index=False)
print("Saved to submission.csv")

Formatting text data...
Starting inference with batch_size=16...


100%|██████████| 1/1 [00:00<00:00,  9.97it/s]

        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.295894        0.295948    0.408158
1   211333        0.398812        0.389716    0.211472
2  1233961        0.307253        0.318286    0.374461
Saved to submission.csv


In [13]:
import torch
from tqdm import tqdm

def predict_batched(model, tokenizer, df, batch_size=16):
    # 1. Auto-detect the device the model is currently on
    # This prevents mismatch errors if model is on 'cuda:0' and you send data to 'cuda:1' or 'cpu'
    device = next(model.parameters()).device
    print(f"Model is on device: {device}")

    model.eval()
    predictions = []

    # 2. Pre-format all text
    print("Formatting text data...")
    formatted_texts = []
    for _, row in df.iterrows():
        # Ensure process_text is defined, otherwise use simple string handling
        prompt = row['prompt'] # process_text(row['prompt']) if you have that function
        res_a = row['response_a']
        res_b = row['response_b']

        formatted_texts.append(
            f"Prompt: {prompt}\n\nResponse A: {res_a}\n\nResponse B: {res_b}"
        )

    # 3. Process in batches
    print(f"Starting inference with batch_size={batch_size}...")
    with torch.no_grad():
        for i in tqdm(range(0, len(formatted_texts), batch_size)):
            batch_texts = formatted_texts[i : i + batch_size]

            # Tokenize
            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=512
            )

            # --- THE FIX: Move inputs to the SAME device as the model ---
            # This handles the dictionary structure safely
            inputs = {key: val.to(device) for key, val in inputs.items()}

            outputs = model(**inputs)
            logits = outputs.logits

            # Move results back to CPU for saving
            preds = torch.argmax(logits, dim=-1).cpu().numpy().tolist()
            predictions.extend(preds)

    return predictions

# Usage
predictions = predict_batched(model, tokenizer, test_df, batch_size=16)

# Map and View
label_map = {0: "winner_model_a", 1: "winner_model_b", 2: "tie"}
test_df['predicted_label'] = [label_map[p] for p in predictions]
test_df[['id', 'predicted_label']].head()

Model is on device: cuda:0
Formatting text data...
Starting inference with batch_size=16...


100%|██████████| 1/1 [00:00<00:00, 10.52it/s]


,id,predicted_label
0,136060,tie
1,211333,winner_model_a
2,1233961,tie


In [14]:
test_df[['id', 'predicted_label']].to_csv("submission.csv", index=False)

In [16]:
test_df.shape

(3, 5)